# Silver — Canais
Remove duplicatas completas, converte o ID para inteiro e padroniza os textos.
Valida IDs e nomes únicos, os quatro canais do cadastro, tipos e status antes de gravar.
Tipos válidos: autosservico e humano. Status válidos: ativo e inativo.

In [0]:
from config import BRONZE_SCHEMA, SILVER_SCHEMA
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

canais_bronze = spark.table(f"{BRONZE_SCHEMA}.canais")

In [0]:
canais_silver = (
    canais_bronze.drop('ingestion_datetime').dropDuplicates()
    .withColumn('id_canal', F.expr('try_cast(id_canal AS INT)'))
)
for coluna in ['nome_canal', 'tipo_canal', 'status_canal']:
    canais_silver = canais_silver.withColumn(
        coluna, F.lower(F.trim(F.regexp_replace(F.col(coluna), r'\s+', ' ')))
    )
canais_silver = canais_silver.withColumn('tipo_canal', F.translate('tipo_canal', 'ç', 'c'))

In [0]:
for coluna in ['id_canal', 'nome_canal']:
    _, passou = dq_check_unique('canais_silver', f'{coluna} único', canais_silver, [coluna])
    if not passou:
        raise ValueError(f'Existem registros diferentes com o mesmo {coluna}.')

regras = {
    'id válido': F.col('id_canal') > 0,
    'nome válido': F.col('nome_canal').isin('ura', 'chatbot', 'atendimento inicial', 'atendimento especializado'),
    'tipo válido': F.col('tipo_canal').isin('autosservico', 'humano'),
    'status válido': F.col('status_canal').isin('ativo', 'inativo'),
}
for nome, condicao in regras.items():
    _, passou = dq_check('canais_silver', nome, canais_silver, condicao)
    if not passou:
        raise ValueError(f'Falha na validação de canais: {nome}')

In [0]:
display(canais_silver)
canais_silver.write.format('delta').mode('overwrite').saveAsTable(f"{SILVER_SCHEMA}.canais")